# 02 — Combine Tracking EDA & Feature Extraction
## NFL Big Data Bowl 2027

**Purpose**: Explore the 10 Hz Combine tracking data in depth. Characterize kinematic trajectories, speed profiles, first-step acceleration, and movement efficiency by drill type, then extract the complete candidate feature library.

**Outputs**:
- `outputs/figures/` — Kinematic and trajectory plots
- `data/processed/combine_features.parquet` — Per-attempt movement features
- `data/processed/player_master.parquet` — Master prospect table

> **Environment**: Compatible with local and Kaggle runtimes with automated `/kaggle/input/` detection.

In [ ]:
# ── Environment & Path Setup ────────────────────────────────────────────────
import sys
import os
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

for p in [Path("."), Path(".."), Path("/kaggle/working"), Path("/kaggle/working/kaggle_notebooks")]:
    if p.exists() and str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

try:
    from src.config import (
        ensure_dirs, FIGURES_DIR, DATA_PROCESSED, COMBINE_FEATURES_PARQUET,
        PLAYER_MASTER_PARQUET, IN_KAGGLE, DATA_RAW
    )
    from src.data_loading import load_players, load_combine_results, load_combine_tracking
    from src.player_mapping import assign_position_groups, build_master_player_table
    from src.combine_features import extract_features_from_tracking, aggregate_player_features
    from src.visualization import (
        plot_drill_player_counts, plot_drill_duration_distribution,
        plot_speed_distribution_by_drill, plot_acceleration_distribution,
        plot_movement_trajectory, plot_position_group_profiles
    )
except ImportError:
    IN_KAGGLE = Path("/kaggle").exists()
    PROJECT_ROOT = Path("/kaggle/working") if IN_KAGGLE else Path(".").resolve()
    DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
    FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
    COMBINE_FEATURES_PARQUET = DATA_PROCESSED / "combine_features.parquet"
    PLAYER_MASTER_PARQUET = DATA_PROCESSED / "player_master.parquet"
    
    # Locate raw data
    raw_candidates = [
        Path("/kaggle/input/nfl-big-data-bowl-2027"),
        Path("/kaggle/input/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
        Path("./data/raw/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
        Path("./data/raw/nfl-big-data-bowl-2027"),
        Path("./data/raw"),
    ]
    DATA_RAW = raw_candidates[0]
    for c in raw_candidates:
        if (c / "players.csv").exists():
            DATA_RAW = c
            break
    if not (DATA_RAW / "players.csv").exists() and Path("/kaggle/input").exists():
        found = list(Path("/kaggle/input").glob("**/players.csv"))
        if found:
            DATA_RAW = found[0].parent
            
    def ensure_dirs():
        DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
        FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    def load_players():
        return pd.read_csv(DATA_RAW / "players.csv")
    def load_combine_results(cols=None):
        return pd.read_csv(DATA_RAW / "combine_results.csv", usecols=cols)
    def load_combine_tracking():
        return pd.read_csv(DATA_RAW / "combine_tracking.csv")
        
    def assign_position_groups(df):
        pos_col = 'nfl_position' if 'nfl_position' in df.columns else ('position' if 'position' in df.columns else None)
        if not pos_col:
            return df
        mapping = {'WR': 'WR', 'TE': 'TE', 'CB': 'CB', 'FS': 'S', 'SS': 'S', 'S': 'S', 'DE': 'Edge', 'EDGE': 'Edge', 'OLB': 'Edge', 'ILB': 'LB', 'MLB': 'LB', 'LB': 'LB', 'DT': 'DT', 'NT': 'DT', 'T': 'OL', 'OT': 'OL', 'G': 'OL', 'OG': 'OL', 'C': 'OL', 'RB': 'RB'}
        df = df.copy()
        df['position_group'] = df[pos_col].map(mapping).fillna('Other')
        return df
        
    def build_master_player_table(players, combine_results=None):
        if combine_results is not None:
            return players.merge(combine_results, on='nfl_id', how='left')
        return players

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')
ensure_dirs()
print(f"Environment: {'Kaggle' if Path('/kaggle').exists() else 'Local'}")
print(f"Data Directory: {DATA_RAW}")
print(f"Processed Data Directory: {DATA_PROCESSED}")

## 1. Load Raw Datasets

In [ ]:
players = load_players()
players = assign_position_groups(players)
combine_results = load_combine_results(cols=None)
tracking = load_combine_tracking()

print(f'Total Combine Prospects: {len(players):,}')
print(f'Tracking Observations: {len(tracking):,}')
if 'position_group' in players.columns:
    print(f'Position groups: {players["position_group"].value_counts().to_dict()}')

## 2. Drill Volume & Participation

Inspect participant counts, frame lengths, and drill coverage across position groups.

In [ ]:
drill_counts = tracking.groupby('drill_type').agg(
    n_rows=('nfl_id', 'count'),
    n_players=('nfl_id', 'nunique'),
).sort_values('n_players', ascending=False)
display(drill_counts)

# Visualize participant volume
plt.figure(figsize=(10, 4))
sns.barplot(x=drill_counts.index, y=drill_counts['n_players'], palette='Blues_d')
plt.xticks(rotation=45, ha='right')
plt.title('Players per Drill Type')
plt.ylabel('Unique Players')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'drill_player_counts.png', dpi=100)
plt.show()

## 3. Kinematic Profiles: Speed & Acceleration Distributions

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=tracking[tracking['drill_type'].isin(['FORTY_YARD_DASH', 'SHORT_SHUTTLE', 'THREE_CONE_DRILL'])], 
            x='drill_type', y='s', palette='Set2')
plt.title('Speed Distribution by Universal Combine Drill')
plt.ylabel('Speed (yds/s)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'drill_speed_distributions.png', dpi=100)
plt.show()

## 4. Trajectory Exploration

Inspect 2D movement paths for direction-change drills (Short Shuttle & Three Cone).

In [ ]:
# Plot representative trajectories for short shuttle
shuttle_sample = tracking[tracking['drill_type'] == 'SHORT_SHUTTLE']
if not shuttle_sample.empty:
    sample_ids = shuttle_sample['nfl_id'].unique()[:3]
    plt.figure(figsize=(10, 5))
    for pid in sample_ids:
        p_track = shuttle_sample[shuttle_sample['nfl_id'] == pid].sort_values('time')
        plt.plot(p_track['x'], p_track['y'], label=f'Player {pid}', alpha=0.8, linewidth=2)
    plt.title('Short Shuttle Movement Trajectories (10 Hz Tracking)')
    plt.xlabel('X Coordinate (yards)')
    plt.ylabel('Y Coordinate (yards)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'short_shuttle_trajectories.png', dpi=100)
    plt.show()

## 5. Movement Feature Library Extraction

Extract kinematics across attempts: peak speed, burst impulse, direction change rate, deceleration, and re-acceleration.

In [ ]:
# Check if precomputed features are available or compute them from tracking
features_df = None

if COMBINE_FEATURES_PARQUET.exists():
    print(f'Loading existing features from {COMBINE_FEATURES_PARQUET}...')
    features_df = pd.read_parquet(COMBINE_FEATURES_PARQUET)
else:
    print('Extracting movement features from 10 Hz tracking...')
    try:
        from src.combine_features import extract_features_from_tracking
        features_df = extract_features_from_tracking(tracking, verbose=True)
    except ImportError:
        # Lightweight robust inline extraction
        records = []
        for (pid, dtype, att), grp in tracking.groupby(['nfl_id', 'drill_type', 'attempt']):
            s = grp['s'].values
            a = grp['a'].values if 'a' in grp.columns else np.zeros_like(s)
            dur = len(grp) * 0.1
            rec = {
                'nfl_id': pid,
                'drill_type': dtype,
                'attempt': att,
                'mean_speed': float(np.mean(s)),
                'max_speed': float(np.max(s)),
                'first_step_acceleration': float(a[:5].mean()) if len(a) >= 5 else 0.0,
                'burst_impulse_0_5s': float(s[:5].mean()) if len(s) >= 5 else 0.0,
                'burst_impulse_0_1s': float(s[:1].mean()) if len(s) >= 1 else 0.0,
                'duration_s': dur,
            }
            if 'dir' in grp.columns and len(grp) > 1:
                d = np.abs(np.diff(grp['dir'].values))
                d = np.where(d > 180, 360 - d, d)
                rec['total_direction_change'] = float(np.sum(d))
                rec['direction_change_rate'] = float(np.sum(d) / max(dur, 0.1))
            else:
                rec['total_direction_change'] = 0.0
                rec['direction_change_rate'] = 0.0
            records.append(rec)
        features_df = pd.DataFrame(records)

    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    features_df.to_parquet(COMBINE_FEATURES_PARQUET, index=False)
    print(f'Successfully saved features to: {COMBINE_FEATURES_PARQUET}')

print(f'Extracted shape: {features_df.shape}')
display(features_df.head())

## 6. Aggregate to Player Level & Build Master Table

In [ ]:
try:
    from src.combine_features import aggregate_player_features
    player_features = aggregate_player_features(features_df, agg_strategy='best_attempt')
except ImportError:
    player_features = features_df.sort_values('max_speed', ascending=False).drop_duplicates(['nfl_id', 'drill_type'])

# Merge prospect master metadata
master = build_master_player_table(players, combine_results=combine_results)
master.to_parquet(PLAYER_MASTER_PARQUET, index=False)
print(f'Player Master saved to: {PLAYER_MASTER_PARQUET} (Shape: {master.shape})')

# Inspect joined player features
pos_map = players[['nfl_id', 'position_group']].drop_duplicates()
feat_with_pos = player_features.merge(pos_map, on='nfl_id', how='left')
print(f'Combined feature table: {feat_with_pos.shape}')
display(feat_with_pos.head())